In [ ]:
import os
import ctypes
import sys
from pathlib import Path

root = Path.cwd()

if sys.platform.startswith('linux'):
    lib_dir = root / 'pathlib' / 'lib_lnx'
    lib_name = 'pathwrap.so'
elif sys.platform == 'darwin':
    lib_dir = root / 'pathlib' / 'lib_osx'
    lib_name = 'pathwrap.dylib'
elif sys.platform.startswith('win'):
    lib_dir = root / 'pathlib' / 'lib_win'
    lib_name = 'pathwrap.dll'
else:
    lib_dir = root / 'pathlib' / 'lib_lnx'
    lib_name = 'pathwrap.so'

pathwrap_path = root / lib_name

if lib_dir.exists():
    ld_path = os.environ.get('LD_LIBRARY_PATH', '')
    if str(lib_dir) not in ld_path:
        os.environ['LD_LIBRARY_PATH'] = f"{lib_dir}:{ld_path}" if ld_path else str(lib_dir)

if not os.environ.get('PATH_LICENSE_STRING'):
    os.environ['PATH_LICENSE_STRING'] = '1259252040&Courtesy&&&USR&GEN2035&5_1_2026&1000&PATH&GEN&31_12_2035&0_0_0&6000&0_0'

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def load_training_stats(stats_path):
    with open(stats_path, "rb") as f:
        return pickle.load(f)

def summarize_rewards(stats):
    rewards = stats["rewards"]
    last_n = min(stats.get("last_n", 1000), len(rewards[0]))
    rows = []

    for agent_id, agent_rewards in enumerate(rewards, start=1):
        data = np.asarray(agent_rewards, dtype=float)
        last_rewards = data[-last_n:] if last_n else data
        rows.append({
            "Scenario": stats.get("scenario_name", "Scenario"),
            "Agent": f"Agent {agent_id}",
            "Mean": float(np.mean(data)),
            "Std": float(np.std(data)),
            "MeanLastN": float(np.mean(last_rewards)),
            "StdLastN": float(np.std(last_rewards)),
            "LastN": int(last_n),
            "Episodes": int(len(data)),
        })

    return rows

def print_summary_table(rows):
    if not rows:
        print("No rows to display.")
        return

    last_n = rows[0]["LastN"]
    headers = [
        "Scenario",
        "Agent",
        "Mean",
        "Std",
        f"Mean (Last {last_n})",
        f"Std (Last {last_n})",
        "Episodes",
    ]
    table_rows = [
        [
            row["Scenario"],
            row["Agent"],
            f'{row["Mean"]:.2f}',
            f'{row["Std"]:.2f}',
            f'{row["MeanLastN"]:.2f}',
            f'{row["StdLastN"]:.2f}',
            str(row["Episodes"]),
        ]
        for row in rows
    ]
    widths = [len(header) for header in headers]

    for row in table_rows:
        widths = [max(width, len(value)) for width, value in zip(widths, row)]

    header_line = " | ".join(header.ljust(width) for header, width in zip(headers, widths))
    separator_line = "-+-".join("-" * width for width in widths)
    print(header_line)
    print(separator_line)
    for row in table_rows:
        print(" | ".join(value.ljust(width) for value, width in zip(row, widths)))

def plot_results(stats, out_path=None):
    rewards = stats["rewards"]
    scenario_name = stats.get("scenario_name", "Scenario")
    window_size = stats.get("window_size", 25)
    separator = stats.get("separator", 10)
    last_n = min(stats.get("last_n", 1000), len(rewards[0]))
    fig, axes = plt.subplots(2, 1, figsize=(15, 10), sharex=True)

    for agent_id, ax in enumerate(axes):
        data = np.asarray(rewards[agent_id], dtype=float)
        episodes = np.arange(len(data))
        moving_avg = [float(np.mean(data[i:i + window_size])) for i in range(0, len(data), window_size)]
        episodes_moving_avg = episodes[:len(moving_avg) * window_size:window_size]
        mean_val = float(np.mean(data))
        mean_last_n = float(np.mean(data[-last_n:]))
        std_val = float(np.std(data))
        std_last_n = float(np.std(data[-last_n:]))

        ax.plot(
            episodes[::separator],
            data[::separator],
            label=f"Agent {agent_id + 1}",
            marker="o",
            linestyle="None",
            markersize=3,
        )
        ax.plot(
            episodes_moving_avg,
            moving_avg,
            label=f"Agent {agent_id + 1} Rolling Average",
            color="blue",
            linestyle="-",
        )
        ax.axhline(mean_val, color="red", linestyle="--", label=f"Agent {agent_id + 1} Mean ({mean_val:.2f})")
        ax.plot([], [], " ", label=f"Agent {agent_id + 1} Mean (Last {last_n} episodes): {mean_last_n:.2f}")
        ax.plot([], [], " ", label=f"Agent {agent_id + 1} Std Dev ({std_val:.2f})")
        ax.plot([], [], " ", label=f"Agent {agent_id + 1} Std Dev (Last {last_n} episodes): {std_last_n:.2f}")
        ax.set_title(f"{scenario_name} - Agent {agent_id + 1} Rewards (SRQ)")
        ax.set_xlabel("Episode")
        ax.set_ylabel("Total Reward")
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if out_path:
        fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# Combined summary is assembled in the final cell after all scenarios are loaded.

## Scenario 1: 3×3 Grid — Agent 1 Bottom-Left → Top-Right, Agent 2 Bottom-Right → Top-Left

In [ ]:
scenario1_stats = load_training_stats("training_stats_scenario1.pkl")
plot_results(scenario1_stats, out_path="training_results_scenario1.png")
scenario1_rows = summarize_rewards(scenario1_stats)

In [ ]:
print_summary_table(scenario1_rows)

## Scenario 2: 3×3 Grid — Agent 1 Top-Left → Bottom-Right, Agent 2 Bottom-Right → Top-Left

In [ ]:
scenario2_stats = load_training_stats("training_stats_scenario2.pkl")
plot_results(scenario2_stats, out_path="training_results_scenario2.png")
scenario2_rows = summarize_rewards(scenario2_stats)

In [ ]:
print_summary_table(scenario2_rows)

## Scenario 3: 4×4 Grid — Agent 1 Bottom-Left → Top-Right, Agent 2 Bottom-Right → Top-Left

In [ ]:
scenario3_stats = load_training_stats("training_stats_scenario3.pkl")
plot_results(scenario3_stats, out_path="training_results_scenario3.png")
scenario3_rows = summarize_rewards(scenario3_stats)

In [ ]:
print_summary_table(scenario3_rows)
combined_rows = scenario1_rows + scenario2_rows + scenario3_rows
print("\nCombined Summary")
print_summary_table(combined_rows)

## Comparison Harness Results

These cells load the multi-pair training artifacts produced by the comparison harness in `bimatrix_game.ipynb`.

In [ ]:
from comparison_harness import (
    discover_training_stats,
    load_training_stats,
    plot_training_stats,
    print_summary_table,
    summarize_rewards,
)

comparison_stats_paths = discover_training_stats("comparison_runs")
print(f"Found {len(comparison_stats_paths)} comparison runs.")
comparison_stats_paths

In [ ]:
comparison_rows = []
comparison_stats = [load_training_stats(path) for path in comparison_stats_paths]

for stats in comparison_stats:
    plot_path = Path("comparison_runs") / stats["scenario_key"] / stats["pair_slug"] / "training_plot.png"
    plot_training_stats(stats, out_path=plot_path)
    comparison_rows.extend(summarize_rewards(stats))

print_summary_table(comparison_rows)